# Integrating 8-bit Mamba C Implementation with Python

In this notebook, we’ll load the 8-bit Mamba model implemented in C as a shared library, define Python wrapper functions, and run an example forward pass.

### Step 1: Load Required Libraries

In [1]:
import ctypes
import numpy as np


### Step 2: Load the Shared Library and Define the C Function Signatures

We load the compiled shared library (`libmamba_model.so`) and define the function signatures for `mamba_init` and `mamba_forward` using Python’s `ctypes` library.

In [2]:
# Load the shared library
mamba_lib = ctypes.CDLL('./8bit/libmamba_model.so')  # Adjust path if necessary

# Define argument and return types for the functions
mamba_lib.mamba_init.restype = None
mamba_lib.mamba_forward.argtypes = [
    np.ctypeslib.ndpointer(dtype=np.int8, ndim=3, flags="C_CONTIGUOUS"),  # Input
    np.ctypeslib.ndpointer(dtype=np.int8, ndim=3, flags="C_CONTIGUOUS")   # Output
]
mamba_lib.mamba_forward.restype = None  # No return value


### Step 3: Create Python Wrapper Functions for Initialization and Forward Pass

Define wrapper functions to initialize the model parameters (`mamba_init`) and perform the forward pass (`mamba_forward`).

In [3]:
# Initialize the Mamba model parameters
def mamba_init():
    """Initialize the 8-bit Mamba model parameters."""
    mamba_lib.mamba_init()

# Perform the forward pass
def mamba_forward(input_array, output_shape):
    """
    Perform the forward pass.
    
    Parameters:
        input_array (np.ndarray): The input data of shape (BATCH_SIZE, SEQ_LENGTH, INPUT_DIM)
        output_shape (tuple): The shape of the output array.
    
    Returns:
        np.ndarray: The output data.
    """
    # Ensure the input is a contiguous array of type int8
    input_array = np.ascontiguousarray(input_array, dtype=np.int8)
    
    # Create an empty output array with the correct shape and type
    output_array = np.zeros(output_shape, dtype=np.int8)
    
    # Call the C function
    mamba_lib.mamba_forward(input_array, output_array)
    
    return output_array


### Step 4: Set Up Parameters and Test the Forward Pass

Initialize the model, create sample input data, and run the forward pass to ensure the integration works correctly.

In [86]:
# Set seeds for reproducibility
torch.manual_seed(42)
np.random.seed(42)

In [100]:
import ctypes
import numpy as np

# Load the shared library
mamba_lib = ctypes.CDLL('./8bit/libmamba_model.so')  # Adjust the path if necessary

# Initialize mamba (if your library has an init function)
mamba_lib.mamba_init()

# Set the argument and return types for functions (adjust based on your actual function definitions)
# For example, if your C function is void mamba_forward(int8_t input[BATCH_SIZE][SEQ_LENGTH][INPUT_DIM],
#                                                      int8_t output[BATCH_SIZE][SEQ_LENGTH][OUTPUT_DIM])
# you need to match these types in ctypes
mamba_lib.mamba_forward.argtypes = [ctypes.POINTER(ctypes.c_int8), ctypes.POINTER(ctypes.c_int8)]
mamba_lib.mamba_forward.restype = None  # Assuming the function returns void

# Example data setup for inputs and outputs (based on your C definitions)
BATCH_SIZE = 3
SEQ_LENGTH = 2
INPUT_DIM = 4
OUTPUT_DIM = 4

# Define input and output arrays in Python, using numpy for easy manipulation
# random data for example
input_data = np.random.randint(0, 50, (BATCH_SIZE, SEQ_LENGTH, INPUT_DIM), dtype=np.int8)
output_data = np.zeros((BATCH_SIZE, SEQ_LENGTH, OUTPUT_DIM), dtype=np.int8)

# Convert numpy arrays to ctypes pointers
input_ptr = input_data.ctypes.data_as(ctypes.POINTER(ctypes.c_int8))
output_ptr = output_data.ctypes.data_as(ctypes.POINTER(ctypes.c_int8))

# Call the mamba_forward function
mamba_lib.mamba_forward(input_ptr, output_ptr)

# Print the output results
print("input_data:")
print(input_data)
print("\noutput_data:")
print("Output from mamba_forward:")
print(output_data)


input_data:
[[[30 46 47 43]
  [14 45 24 19]]

 [[ 7 49 39 22]
  [13 29  8 18]]

 [[22 46 43 36]
  [21  5  3 20]]]

output_data:
Output from mamba_forward:
[[[ 0  0 13 13]
  [ 0  0  0  0]]

 [[ 0  0  0  0]
  [ 0  0  0  0]]

 [[ 0  0  0  0]
  [ 0  0  0  0]]]


In [88]:
from mamba_ssm import Mamba


In [95]:
import torch
from mamba_ssm import Mamba  # Adjust this import path as needed

# Define model parameters based on your setup
d_model = 4       # Matches INPUT_DIM
d_state = 16      # State expansion factor (can be adjusted)
d_conv = 1        # Minimal convolution width
expand = 1       # Minimal expansion factor

# Initialize the Mamba model
model = Mamba(
    d_model=d_model,
    d_state=d_state,
    d_conv=d_conv,
    expand=expand
).to("cuda")  # Move to GPU if available


In [96]:
with torch.no_grad():
    for param in model.parameters():
        torch.nn.init.constant_(param, 0.1)  # Example: Initialize all weights to 0.01


In [97]:
input_data_torch = torch.tensor(input_data, dtype=torch.float32).to("cuda")

In [98]:
output_data = model.forward(input_data_torch)
print("Output from Mamba model:")
print(output_data)

Output from Mamba model:
tensor([[[ 3.2931,  3.2931,  3.2931,  3.2931],
         [11.5819, 11.5819, 11.5819, 11.5819]],

        [[16.7013, 16.7013, 16.7013, 16.7013],
         [ 8.4424,  8.4424,  8.4424,  8.4424]],

        [[ 6.8728,  6.8728,  6.8728,  6.8728],
         [ 2.2496,  2.2496,  2.2496,  2.2496]],

        [[ 1.1270,  1.1270,  1.1270,  1.1270],
         [22.8418, 22.8418, 22.8418, 22.8418]],

        [[19.7308, 19.7308, 19.7308, 19.7308],
         [26.3594, 26.3594, 26.3594, 26.3594]],

        [[ 0.6229,  0.6229,  0.6229,  0.6229],
         [ 4.7160,  4.7160,  4.7160,  4.7160]],

        [[ 1.1919,  1.1919,  1.1919,  1.1919],
         [ 0.2561,  0.2561,  0.2561,  0.2561]],

        [[15.0697, 15.0697, 15.0697, 15.0697],
         [ 8.8194,  8.8194,  8.8194,  8.8194]],

        [[ 3.2931,  3.2931,  3.2931,  3.2931],
         [ 0.3969,  0.3969,  0.3969,  0.3969]],

        [[ 2.4746,  2.4746,  2.4746,  2.4746],
         [ 7.8747,  7.8747,  7.8747,  7.8747]],

        [[ 1.56